# Advanced Model: Deep Learning (1D-CNN + LSTM)

Unser Feature-Engineering mit LightGBM hat uns auf **0.7002** gebracht. Um **Baseline 3 (0.7088)** sicher und deutlich zu schlagen, zünden wir jetzt die nächste Stufe: Deep Learning.

Bei Beschleunigungs- und Zeitreihendaten sind **Neuronale Netze** der Goldstandard. Wir bauen hier:
1. Ein **1D-CNN (Convolutional Neural Network)**: Dies lernt automatisch lokale Muster in der Zeitreihe (wie kleine Schock-Wellen beim Gehen).
2. Ein **Bi-direktionales LSTM**: Dieses lernt die langfristigen Abhängigkeiten über die ganzen 300 Sekunden hinweg (z. B. "erst lag er ruhig, dann stand er auf").

**WICHTIG**: Aktiviere rechts in Kaggle unter "Notebook options" unbedingt den **GPU-Accelerator (P100 oder T4x2)**, sonst dauert das Training ewig!

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score
import warnings
warnings.filterwarnings('ignore')

# Dieser Befehl verhindert den 'CUDA error: no kernel image' Bug, 
# der auf Kaggle oft bei LSTMs auf T4 GPUs auftritt.
torch.backends.cudnn.enabled = False

## 1. Device Setup & Daten Laden

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Nutze Gerät: {device}")
if device.type == 'cpu':
    print("WARNUNG: Keine GPU gefunden! Bitte in Kaggle rechts 'Accelerator' auf 'GPU' stellen.")

KAGGLE_PATH = '/kaggle/input/datasets/axxtur/nycu-data-mining-assignment-3'
if not os.path.exists(KAGGLE_PATH):
    KAGGLE_PATH = '/kaggle/input/nycu-data-mining-assignment-3'
    if not os.path.exists(KAGGLE_PATH):
        KAGGLE_PATH = 'nycu-data-mining-assignment-3'

print("Lade Daten...")
train_data = np.load(os.path.join(KAGGLE_PATH, 'train_data.npz'), allow_pickle=True)
X_train_raw = train_data['X']
y_train = train_data['y']
file_ids_train = train_data['file_ids']
user_ids_train = train_data['user_ids']

test_data = np.load(os.path.join(KAGGLE_PATH, 'test_data.npz'), allow_pickle=True)
X_test_raw = test_data['X']
file_ids_test = test_data['file_ids']

# Skalierung (Neuronale Netze hassen unskalierte Daten)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw.reshape(-1, 6)).reshape(X_train_raw.shape)
X_test_scaled = scaler.transform(X_test_raw.reshape(-1, 6)).reshape(X_test_raw.shape)


## 2. Dataset und Modell-Architektur definieren

In [ ]:
class HARDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long) if y is not None else None
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        # PyTorch Conv1d erwartet Shape (Batch, Channels, Length)
        x_item = self.X[idx].permute(1, 0)
        if self.y is not None:
            return x_item, self.y[idx]
        return x_item

class HARModel(nn.Module):
    def __init__(self, num_classes=6):
        super(HARModel, self).__init__()
        
        # 1D-CNN Schichten
        self.conv1 = nn.Conv1d(in_channels=6, out_channels=64, kernel_size=9, padding=4)
        self.bn1 = nn.BatchNorm1d(64)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(kernel_size=2)
        
        self.conv2 = nn.Conv1d(64, 128, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(128)
        
        self.conv3 = nn.Conv1d(128, 256, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(256)
        
        # LSTM Schicht (Bidirektional)
        self.lstm = nn.LSTM(input_size=256, hidden_size=128, num_layers=1, batch_first=True, bidirectional=True)
        
        # Klassifikator (Dense)
        self.fc = nn.Sequential(
            nn.Linear(128 * 2, 64),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(64, num_classes)
        )
        
    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)
        
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)
        
        x = self.relu(self.bn3(self.conv3(x)))
        x = self.pool(x)
        
        x = x.permute(0, 2, 1)
        
        lstm_out, _ = self.lstm(x)
        x = lstm_out[:, -1, :]
        
        return self.fc(x)

## 3. Training & Validation Loop

In [ ]:
def train_model(model, train_loader, val_loader, epochs=15):
    class_counts = np.bincount(y_train)
    weights = 1.0 / class_counts
    weights = weights / weights.sum() * len(class_counts)
    criterion = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float32).to(device))
    
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)
    
    best_f1 = 0.0
    best_model_weights = None
    
    for epoch in range(epochs):
        model.train()
        train_preds, train_true = [], []
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            
            optimizer.zero_grad()
            outputs = model(X_b)
            loss = criterion(outputs, y_b)
            loss.backward()
            optimizer.step()
            
            train_preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())
            train_true.extend(y_b.cpu().numpy())
            
        model.eval()
        val_preds, val_true = [], []
        with torch.no_grad():
            for X_b, y_b in val_loader:
                X_b, y_b = X_b.to(device), y_b.to(device)
                outputs = model(X_b)
                val_preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())
                val_true.extend(y_b.cpu().numpy())
                
        val_f1 = f1_score(val_true, val_preds, average='macro')
        scheduler.step(val_f1)
        
        print(f"Epoch {epoch+1}/{epochs} | Val F1-Macro: {val_f1:.4f}")
        
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_model_weights = model.state_dict()
            
    model.load_state_dict(best_model_weights)
    return model, best_f1

## 4. Cross-Validation Starten

In [ ]:
gkf = GroupKFold(n_splits=5)
trained_models = []
cv_scores = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(X_train_scaled, y_train, groups=user_ids_train)):
    print(f"\n--- Fold {fold+1} ---")
    train_ds = HARDataset(X_train_scaled[train_idx], y_train[train_idx])
    val_ds = HARDataset(X_train_scaled[val_idx], y_train[val_idx])
    
    train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=128, shuffle=False)
    
    model = HARModel(num_classes=6).to(device)
    
    best_model, fold_f1 = train_model(model, train_loader, val_loader, epochs=15)
    
    trained_models.append(best_model)
    cv_scores.append(fold_f1)
    print(f"-> Fold {fold+1} Best F1-Macro: {fold_f1:.4f}")

print(f"\nOverall Cross-Validation F1-Macro: {np.mean(cv_scores):.4f}")

## 5. Test Predictions und Submission

In [ ]:
test_ds = HARDataset(X_test_scaled)
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False)

test_preds_proba = np.zeros((len(X_test_scaled), 6))

for model in trained_models:
    model.eval()
    fold_preds = []
    with torch.no_grad():
        for X_b in test_loader:
            X_b = X_b.to(device)
            outputs = torch.softmax(model(X_b), dim=1)
            fold_preds.extend(outputs.cpu().numpy())
            
    test_preds_proba += np.array(fold_preds) / len(trained_models)

final_preds = np.argmax(test_preds_proba, axis=1)

submission = pd.DataFrame({
    'Id': file_ids_test,
    'Label': final_preds
})

submission.to_csv('submission_dl.csv', index=False)
print("Saved submission_dl.csv! Ready for Kaggle upload.")
submission.head()